In [26]:
import numpy as np
import pandas as pd
import time

from lib.envs.maze import MazeEnv

In [27]:
class RL():
    def __init__(self, action_space, 
                 learning_rate=0.01, 
                 reward_decay =0.9, 
                 e_greedy     =0.95):
        
        self.actions = action_space
        self.lr = learning_rate
        self.gamma = reward_decay
        self.epsilon = e_greedy
        self.q_table = pd.DataFrame(columns=self.actions, dtype=np.float64)
        
    def check_state_exist(self, state):
        if state not in self.q_table.index:
            # 如果状态在当前的 Q 表中不存在,将当前状态加入 Q 表中
            new_row = pd.DataFrame(
                [[0.0] * len(self.actions)],
                columns=self.q_table.columns,
                index=[state]
            )
            self.q_table = pd.concat([self.q_table, new_row])

    def choose_action(self, observation):
        self.check_state_exist(observation)
        # 从均匀分布的[0,1]中随机采样,当小于阈值时采用选择最优行为的方式,
        # 当大于阈值时采用选择随机行为的方式,这样人为增加随机性是为了解决陷入局部最优
        
        # 最优行为概率：ε + (1-ε)/|A|
        # 其它行为概率：(1-ε)/|A|
        if np.random.rand() <= self.epsilon:
            # 选择最优行为
            state_action = self.q_table.loc[observation, :]
            # 因为一个状态下最优行为可能有多个,需要随机选择一个
            max_val = state_action.max()
            best_actions = state_action[state_action == max_val].index.tolist()
            action = np.random.choice(best_actions)
        else:
            # 选择随机行为
            action = np.random.choice(self.actions)
        return action

    def learn(self, *args):
        pass

In [28]:
# 离线策略 Q-Learning
class QLearningTable(RL):  # 继承 RL 基类
    def __init__(self, actions, 
                 learning_rate=0.01, 
                 reward_decay =0.9, 
                 e_greedy     =0.9):
        super().__init__(actions, learning_rate, reward_decay, e_greedy)

    def learn(self, s, a, r, s_):
        self.check_state_exist(s)
        self.check_state_exist(s_)
        
        q_predict = self.q_table.loc[s, a]

        if s_ != 'terminal':
            # 使用公式: Q_target = r + γ maxQ(s',a')
            q_target = r + self.gamma * self.q_table.loc[s_, :].max()
        else:
            q_target = r

        # 更新公式: Q(s,a) ← Q(s,a) + α(r + γ max Q(s',a') − Q(s,a))
        self.q_table.loc[s, a] += self.lr * (q_target - q_predict)

    def get_action(self, q_table, state):
        # 选择最优行为
        state_action = q_table.loc[state, :]
        max_val = state_action.max()
        
        idxs = [i for i, v in enumerate(state_action) if v == max_val]
        idxs.sort()
        return tuple(idxs)
    
    def get_policy(self, q_table, env, rows=5, cols=5, pixels=40, origin=20):
        policy = []
        # 预计算所有陷阱和宝藏坐标
        hell_positions = [
            env.canvas.coords(env.hell1), 
            env.canvas.coords(env.hell2),
            env.canvas.coords(env.hell3), 
            env.canvas.coords(env.hell4),
            env.canvas.coords(env.hell5), 
            env.canvas.coords(env.hell6),
            env.canvas.coords(env.hell7), 
            env.canvas.coords(env.oval)
        ]
        
        for i in range(rows):
            for j in range(cols):
                # 求出每个格子的状态
                item_center_x = (j * pixels + origin)
                item_center_y = (i * pixels + origin)
                item_state = [item_center_x - 15.0, 
                              item_center_y - 15.0, 
                              item_center_x + 15.0, 
                              item_center_y + 15.0]
                                
                if item_state in hell_positions:
                    policy.append(-1)
                    continue
                
                if str(item_state) not in q_table.index:
                    policy.append((0, 1, 2, 3))
                    continue
                # 选择最优行为
                item_action_max = self.get_action(q_table, str(item_state))
                policy.append(item_action_max)
        return policy

In [29]:
def update(agent, env, episodes=100, render=True, verbose=True):
    """
    Sarsa 训练主循环
    agent: SarsaTable 实例
    env: MazeEnv 实例
    episodes: 训练局数
    render: 是否渲染训练过程
    verbose: 是否打印进度
    返回: policy_grid, metrics
    """
    metrics = {'rewards': [], 'steps': [], 'success': []}

    start_time = time.perf_counter()

    for episode in range(episodes):
        # 初始化状态
        observation = env.reset(render=render)
        c = 0
        tmp_policy = {}

        ep_reward = 0
        ep_steps = 0
        success = False
        
        while True:
            # 渲染当前环境
            if render:
                env.render()

            # 基于当前状态选择行为
            action = agent.choose_action(str(observation))

            state_item = tuple(observation)
            tmp_policy[state_item] = action

            # 采取行为获得下一个状态和回报及是否终止
            next_obs, reward, done, oval_flag = env.step(action)
            # 根据当前的变化开始更新 Q
            agent.learn(str(observation), action, reward, str(next_obs))

            # 改变状态和行为
            observation = next_obs

            ep_reward += reward
            ep_steps += 1

            c += 1
            # 如果为终止状态,则结束当前的局数
            if done:
                success = oval_flag
                break

        metrics['rewards'].append(ep_reward)
        metrics['steps'].append(ep_steps)
        metrics['success'].append(1 if success else 0)

        if verbose and (episode + 1) % 10 == 0:
            print(f'第 {episode + 1} 局结束')

    stop_time = time.perf_counter()
    print('Duration time is %.3f seconds'%(stop_time - start_time))

    # 开始输出最终的 Q 表
    q_table_result = agent.q_table
    print(q_table_result)

    # 使用 Q 表输出各状态的最优策略
    policy = agent.get_policy(q_table_result, env)
    policy_grid = np.array(policy, dtype=object).reshape(5,5)

    if verbose:
        print('\n最优策略 (0:↑ 1:↓ 2:← 3:→,  -1:障碍/宝藏):')
        print(policy_grid)
        env.render_by_policy(policy_grid)

    return policy_grid, metrics



In [30]:
env = MazeEnv()
agent = QLearningTable(actions=list(range(env.n_actions))) 
env.after(100, lambda: update(agent, env, episodes=100, render=False, verbose=True))

# env.destroy()
env.mainloop()

第 10 局结束
第 20 局结束
第 30 局结束
第 40 局结束
第 50 局结束
第 60 局结束
第 70 局结束
第 80 局结束
第 90 局结束
第 100 局结束
Duration time is 0.888 seconds
                                     0         1         2         3
[5.0, 5.0, 35.0, 35.0]       -0.924157 -0.920359 -0.917317 -0.923954
[5.0, 45.0, 35.0, 75.0]      -0.717184 -0.714689 -0.721203 -0.956179
[5.0, 85.0, 35.0, 115.0]     -0.542428 -0.540647 -0.550378 -0.543441
[45.0, 85.0, 75.0, 115.0]    -0.394040 -0.490100 -0.327557 -0.313782
terminal                      0.000000  0.000000  0.000000  0.000000
[-35.0, 5.0, -5.0, 35.0]     -0.823905 -0.820940 -0.824223 -0.828943
[45.0, 5.0, 75.0, 35.0]      -0.750440 -0.956179 -0.758245 -0.756682
[-35.0, 45.0, -5.0, 75.0]    -0.724172 -0.720660 -0.722693 -0.714447
[-35.0, 85.0, -5.0, 115.0]   -0.608910 -0.604955 -0.601182 -0.602684
[85.0, 5.0, 115.0, 35.0]     -0.610577 -0.679347 -0.609564 -0.613462
[125.0, 5.0, 155.0, 35.0]    -0.487944 -0.585199 -0.489717 -0.489525
[-35.0, 125.0, -5.0, 155.0]  -0.535700 -0.530684 -